# 02 · Classification — interactive walkthrough

Companion to the [report](report.qmd): supervised vs zero-shot, then the zero-shot
super-power — classifying against *arbitrary* labels with no retraining.

```bash
uv sync --group dev
uv run jupyter lab   # open modules/02-classification/explore.ipynb
```

In [ ]:
import matplotlib.pyplot as plt
from cvkit.devices import get_device, describe_device
from cvkit.io import load_image
import classify

device = get_device()
print(describe_device(device))

## An Imagenette image

In [ ]:
import torchvision
ds = torchvision.datasets.Imagenette('data/imagenette', split='val', size='320px', download=True)
path, label = ds._samples[1500]
img = load_image(path)
plt.imshow(img); plt.axis('off'); plt.title(f'truth: {classify.CLASS_NAMES[label]}'); plt.show()

## Supervised vs zero-shot on the same image
All three adapters share a `scores(image) -> 10 values` interface.

In [ ]:
for key in ['convnext-tiny', 'siglip2-base', 'openclip-vitb32']:
    m = classify.build_classifier(key, device)
    pred = classify.CLASS_NAMES[m.predict(img)]
    print(f'{m.meta.display:20s} [{m.meta.paradigm:10s}] -> {pred}')

## The zero-shot super-power: classify against *any* labels
No retraining — just change the strings. Try labels that aren't Imagenette classes at all.

In [ ]:
import open_clip, torch
from PIL import Image
m, _, pre = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
m = m.to(device).eval(); tok = open_clip.get_tokenizer('ViT-B-32')

labels = ['a fish', 'a dog', 'a musical instrument', 'a vehicle', 'a building', 'sports equipment']
with torch.no_grad():
    tf = m.encode_text(tok([f'a photo of {l}' for l in labels]).to(device)); tf /= tf.norm(dim=-1, keepdim=True)
    ie = m.encode_image(pre(Image.fromarray(img)).unsqueeze(0).to(device)); ie /= ie.norm(dim=-1, keepdim=True)
    sims = (ie @ tf.T)[0]
for l, s in sorted(zip(labels, sims.tolist()), key=lambda x: -x[1]):
    print(f'{s:.3f}  {l}')